# Estimating behavioural equations

A ModelFlow model combines **identities**, which hold exactly, with **behavioural equations**, whose coefficients are estimated from data. This chapter shows how to estimate behavioural equations with **`Estimate_nls`** — non-linear least squares based on the `lmfit` package — both stand-alone and as part of building a model with `Makemodel` and the `%%Makemymodel` magic.

`Estimate_nls` is a factory class: with the default `solver='lmfit'` it returns an `Estimate_nls_lmfit` instance, which is the backend covered here.

In [1]:
from modelclass import model
import modeljupytermagic                     # %%dataframe and %%Makemymodel magics
from modelconstruct_estimation import Makemodel
from modelestimator_new import Estimate_nls

# Development support
%load_ext autoreload
%autoreload 2

## Data

Estimation needs data in a Pandas `DataFrame` with time on the index. To keep this chapter self-contained the data — from a World Bank model for Nepal — is entered directly with the `%%dataframe` magic.

In [2]:
%%dataframe df start=2010
              Y     CON    GOV     INV     EXP     IMP
2010	1507633     nan    nan  376664     nan     nan
2011	1559222	1320302	127814	373939	121715	444232
2012	1632040	1360376	128860	381170	146293	457746
2013	1689572	1396402	124255	414557	165235	522448
2014	1791141	1438904	138533	466473	194706	632208
2015	1862357	1476067	154467	536417	199215	692795
2016	1870424	1537410	135991	570679	164739	714626
2017	2038337	1549515	165119	702408	179327	916470
2018	2193706	1645118	168507	785371	193125	1090956
2019	2339743	1779022	184955	874481	203831	1154398
2020	2284300	1842997	192011	796389	171458	913728
2021	2394818	1989827	188832	874412	134904	1085637
2022	2529243	2125755	207023	907774	180852	1249408
2023	2576251	2144255	202054	809228	190884	1034756
2024	2737770	2187140	213861	847178	223748	998743

In [3]:
npl = df.mfcalc('''
ydisc = y-(con+inv+gov+eXp-imp)
gde   = (con+gov+inv)
''')
npl.index = npl.index.year

In [4]:
var_description = {
'Y':'GDP',
'CON':'Private Consumption',
'GOV':'Government',
'INV':'Investment',
'EXP':'Exports',
'IMP':'Imports',
'YDISC':'Statistical Discrepancy',
'GDE':'Domestic Demand',
}

## A default estimator with `.with_defaults()`

Most estimations in a model share the same data, sample and naming conventions. The `.with_defaults()` factory collects these once and returns a callable — here named `ls` — that constructs and runs an estimation for any equation it is given. Any keyword passed to the callable overrides the stored default.

The `param_names=` option deserves attention: by default coefficients are written `C(1)`, `C(2)`, ... but declaring `param_names=['lambda', 'beta_shortterm', 'beta_longterm']` allows the self-documenting placeholders `LAMBDA`, `BETA_SHORTTERM(10)` and `BETA_LONGTERM(1)` instead. A bare name like `lambda` is shorthand for placeholder number 1 of that prefix.

In [5]:
ls = Estimate_nls.with_defaults(
    smpl            = (2011, 2019),   # estimate on the pre-covid period
    input_df        = npl,
    var_description = var_description,
    param_names     = ['lambda', 'beta_shortterm', 'beta_longterm'],
)

## Estimating a single equation

The equation below is a non-linear ECM for private consumption:

$$ \Delta c_t = - \lambda \,(c_{t-1}- y_{t-1} - \log\beta^{longterm}_1 ) + \beta^{shortterm}_{10}\, \Delta y_t + \epsilon_t $$

where lower-case letters are logarithms. Calling `ls(...)` parses the equation, runs the lmfit minimizer on the residuals, and returns an estimator object. Displayed on its own, the object renders a full HTML report with the regression results and an actual-versus-fitted plot.

In [6]:
est_con = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)",
             caption='Private consumption')
est_con

Estimate_nls_lmfit(org_eq='DLOG(CON) = -LAMBDA__1*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS(BETA_LONGTERM__1))) + BETA_SHORTTERM__10*DLOG(Y)', smpl=(2011, 2019), input_df=               Y        CON       GOV       INV       EXP        IMP  \
index                                                                  
2010   1507633.0        NaN       NaN  376664.0       NaN        NaN   
2011   1559222.0  1320302.0  127814.0  373939.0  121715.0   444232.0   
2012   1632040.0  1360376.0  128860.0  381170.0  146293.0   457746.0   
2013   1689572.0  1396402.0  124255.0  414557.0  165235.0   522448.0   
2014   1791141.0  1438904.0  138533.0  466473.0  194706.0   632208.0   
2015   1862357.0  1476067.0  154467.0  536417.0  199215.0   692795.0   
2016   1870424.0  1537410.0  135991.0  570679.0  164739.0   714626.0   
2017   2038337.0  1549515.0  165119.0  702408.0  179327.0   916470.0   
2018   2193706.0  1645118.0  168507.0  785371.0  193125.0  1090956.0   
2019   2339743.0  1779022.0  184955.0  874481.0  203831.0  1154398.0   
2020   2284300.0  1842997.0  192011.0  796389.0  171458.0   913728.0   
2021   2394818.0  1989827.0  188832.0  874412.0  134904.0  1085637.0   
2022   2529243.0  2125755.0  207023.0  907774.0  180852.0  1249408.0   
2023   2576251.0  2144255.0  202054.0  809228.0  190884.0  1034756.0   
2024   2737770.0  2187140.0  213861.0  847178.0  223748.0   998743.0   

             GDE     YDISC  
index                       
2010         NaN       NaN  
2011   1822055.0   59684.0  
2012   1870406.0   73087.0  
2013   1935214.0  111571.0  
2014   2043910.0  184733.0  
2015   2166951.0  188986.0  
2016   2244080.0  176231.0  
2017   2417042.0  358438.0  
2018   2598996.0  492541.0  
2019   2838458.0  451852.0  
2020   2831397.0  195173.0  
2021   3053071.0  292480.0  
2022   3240552.0  357247.0  
2023   3155537.0  264586.0  
2024   3248179.0  264586.0  , est_param='C', param_names=['lambda', 'beta_shortterm', 'beta_longterm'], caption='Private consumption', var_description={'Y': 'GDP', 'CON': 'Private Consumption', 'GOV': 'Government', 'INV': 'Investment', 'EXP': 'Exports', 'IMP': 'Imports', 'YDISC': 'Statistical Discrepancy', 'GDE': 'Domestic Demand'}, omodel=DummyOModel(var_description={'Y': 'GDP', 'CON': 'Private Consumption', 'GOV': 'Government', 'INV': 'Investment', 'EXP': 'Exports', 'IMP': 'Imports', 'YDISC': 'Statistical Discrepancy', 'GDE': 'Domestic Demand'}), coef_dict={}, constraints={}, frml_name='<STOC,DAMP>', add_add_factor=True, make_fixable=True, make_fitted=False, ecm=True, fit_kws={}, method='least_squares', regression_model=<lmfit.minimizer.MinimizerResult object at 0x0000018C5CD2E270>, coef_estimate_dict={'BETA_LONGTERM__1': 0.8842598154044378, 'BETA_SHORTTERM__10': -0.09711177512345369, 'LAMBDA__1': 0.44166994168701856}, org_eq_baked='DLOG(CON) = -(0.4416699417)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.8842598154)))) + (-0.0971117751)*DLOG(Y)', mfresult=LSResult(olsmodel=...), coef_ser=BETA_LONGTERM__1      0.884260
BETA_SHORTTERM__10   -0.097112
LAMBDA__1             0.441670
Name: Private consumption, dtype: float64, default_params={}, lmfit_params=Parameters([('BETA_LONGTERM__1', <Parameter 'BETA_LONGTERM__1', value=0.1, bounds=[-inf:inf]>), ('BETA_SHORTTERM__10', <Parameter 'BETA_SHORTTERM__10', value=0.1, bounds=[-inf:inf]>), ('LAMBDA__1', <Parameter 'LAMBDA__1', value=0.1, bounds=[-inf:inf]>)]))

### Inspecting the result

The estimator object exposes the result in several convenient forms:

| Attribute | Content |
|---|---|
| `coef_ser` | Estimated coefficients as a Pandas `Series` |
| `tvalues` | t-statistics as a `Series` (NaN for fixed or derived coefficients) |
| `org_eq_baked` | The equation with the estimated numbers substituted in place |
| `af_df` | Actual and fitted values over the estimation sample |
| `residuals_df` | Residuals over the estimation sample |
| `regression_model` | The native `lmfit` result object |
| `get_html_report()` | The full HTML report as a string |

In [7]:
print(est_con.org_eq_baked)
est_con.coef_ser

DLOG(CON) = -(0.4416699417)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.8842598154)))) + (-0.0971117751)*DLOG(Y)


BETA_LONGTERM__1      0.884260
BETA_SHORTTERM__10   -0.097112
LAMBDA__1             0.441670
Name: Private consumption, dtype: float64

In [8]:
est_con.tvalues

BETA_LONGTERM__1      30.389199
BETA_SHORTTERM__10    -0.482968
LAMBDA__1              3.760746
Name: Private consumption, dtype: float64

## Constraints — the `ST.` clause

Coefficients can be constrained directly in the equation string after the keyword `ST.`, with individual constraints separated by `;`:

    y = c(1) + c(2)*x + alfa*z st. c(1)~0.5; c(2)>0; alfa:=c(1)+c(2)

| Form | Meaning |
|---|---|
| `NAME = value` | Fix the coefficient at `value` (not estimated) |
| `NAME ~ value` | Starting value (still estimated) |
| `NAME > x` / `NAME >= x` | Lower bound |
| `NAME < x` / `NAME <= x` | Upper bound |
| `NAME = [lo, hi]` | Lower and upper bound |
| `NAME := expr` | Derived coefficient: an algebraic function of other coefficients |

The same constraints can be supplied programmatically through the `constraints=` option as per-parameter `lmfit` settings, e.g. `constraints={'LAMBDA__1': {'value': 0.3, 'vary': False}}`.

Below the speed of adjustment $\lambda$ is fixed at 0.3, so only the two $\beta$s are estimated:

In [9]:
est_fix = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y) st. lambda = 0.3",
             caption='Private consumption, lambda fixed')
est_fix.coef_ser

BETA_LONGTERM__1      0.918278
BETA_SHORTTERM__10   -0.053508
LAMBDA__1             0.300000
Name: Private consumption, lambda fixed, dtype: float64

And here $\lambda$ is restricted to the interval $[0.1,\;0.5]$ while the short-run elasticity is kept non-negative:

In [10]:
est_bound = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y) st. lambda = [0.1, 0.5]; beta_shortterm(10) >= 0",
               caption='Private consumption, bounded')
est_bound.coef_ser

BETA_LONGTERM__1      8.761969e-01
BETA_SHORTTERM__10    1.318554e-15
LAMBDA__1             4.314731e-01
Name: Private consumption, bounded, dtype: float64

## Options reference

All options accepted by `Estimate_nls` — directly or through `.with_defaults()`:

| Option | Default | Description |
|---|---|---|
| `org_eq` | — | The equation (positional when calling the factory) |
| `input_df` | `None` | `DataFrame` with the data; time on the index |
| `smpl` | `(2002, 2018)` | Estimation sample, inclusive |
| `est_param` | `'C'` | Prefix for numbered coefficients `C(1)`, `C(2)`, ... |
| `param_names` | `None` | Additional named coefficient prefixes, e.g. `['lambda', 'beta_longterm']` |
| `caption` | `'Estimation of '` | Caption used in reports |
| `var_description` | `{}` | `{variable: description}` used to label output |
| `constraints` | `{}` | Per-parameter lmfit settings, e.g. `{'C__1': {'min': 0, 'max': 1}}`; also filled from an inline `ST.` clause |
| `default_params` | `{}` | Per-parameter starting values and bounds, e.g. `{'C__2': {'value': -0.1, 'min': -1.0, 'max': 0.0}}` |
| `method` | `'least_squares'` | Minimizer method passed to `lmfit.minimize` |
| `fit_kws` | `{}` | Extra keyword arguments forwarded to `lmfit.minimize` |
| `coef_dict` | `{}` | Numeric substitutions applied to placeholders before parsing |
| `frml_name` | `'<STOC,DAMP>'` | FRML tag used when the estimated equation is emitted |
| `add_add_factor` | `True` | Generate an add factor in the normalized equation |
| `make_fixable` | `True` | Generate `_X`/`_D` fixing variables |
| `make_fitted` | `False` | Generate a `_FITTED` equation |

Every parameter without an explicit starting value starts at `0.1`. For ECM equations the speed-of-adjustment often needs help: give it a starting value or bounds through `default_params` or an `ST.` clause.

## Estimation while building a model

Estimation really shines inside `Makemodel`: an equation tagged with `<estimator=...>` is estimated during model construction, and the estimated coefficients are baked into the emitted `FRML`. The estimator name — here `ls` — is resolved from the calling namespace (or from an explicit `estimator_classes=` mapping).

### Estimation tags

| Tag | Effect |
|---|---|
| `<estimator=name>` (alias `<est=name>`) | Estimate this equation with the estimator `name` |
| `<estimator>` / `<est>` | Estimate with the model-wide default passed as `Makemodel(..., estimator=...)` |
| `<stoc>` | Mark the equation as behavioural: implies `<exo>` (fixable) and `<add>` (add factor) |
| `<smpl=start end>` | Equation-local estimation sample; overrides the estimator and `Makemodel` defaults |
| `<caption='text'>` | Caption used for the equation in estimation output and reports |
| `<constraints=...>` | Coefficient constraints, appended to the equation as an `ST.` clause |
| `<drop>` | Skip the equation when the FRMLs are emitted (the text is kept in the documentation) |

In [11]:
eqs = Makemodel('''
## Private consumption
><stoc,estimator=ls,caption='Private consumption'> dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

## Government expenditure
Estimated on a shorter, equation-local sample:
><stoc,estimator=ls,smpl=2012 2019> dlog(gov) = -lambda*(log(gov(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

## GDP identity
> <ident> y = con+inv+gov+exp-imp+ydisc
''')
eqs.show

FRML <STOC,ESTIMATOR=LS,CAPTION='PRIVATE CONSUMPTION'> CON = (CON(-1)*EXP(CON_A+ (-(0.4416699417)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.8842598154))))+(-0.0971117751)*((LOG(Y))-(LOG(Y(-1))))) )) * (1-CON_D)+ CON_X*CON_D  $
FRML <STOC,ESTIMATOR=LS,SMPL=2012 2019> GOV = (GOV(-1)*EXP(GOV_A+ (-(1.2457148604)*(LOG(GOV(-1))-LOG(Y(-1))-LOG(ABS((0.0756258858))))+(1.6947092960)*((LOG(Y))-(LOG(Y(-1))))) )) * (1-GOV_D)+ GOV_X*GOV_D  $
FRML <IDENT> Y = CON+INV+GOV+EXP-IMP+YDISC $

FRML <CALC_ADD_FACTOR>  CON_A = - ((-(0.4416699417)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.8842598154))))+(-0.0971117751)*((LOG(Y))-(LOG(Y(-1)))))) +LOG(CON)-LOG(CON(-1)) $ 
FRML <CALC_ADD_FACTOR>  GOV_A = - ((-(1.2457148604)*(LOG(GOV(-1))-LOG(Y(-1))-LOG(ABS((0.0756258858))))+(1.6947092960)*((LOG(Y))-(LOG(Y(-1)))))) +LOG(GOV)-LOG(GOV(-1)) $


### Estimation output and reports

- `.render_est` renders the model text with estimation tables (coefficients, t-statistics, fit measures) in the notebook — the `%%Makemymodel` magic does this automatically.
- `.tvalues` collects the t-statistics of all estimated equations in a `DataFrame`.
- `.estimation_report(open_file=True)` writes an HTML report with actual-versus-fitted plots for every estimated equation; `report_all=True` also includes the identities.
- `.report().show()` displays the full model documentation — prose, equations and collapsible estimation panels.
- `.showestimation_records` prints the raw record of every estimation run during construction.

In [12]:
eqs.tvalues

,CON,GOV
BETA_LONGTERM__1,30.389199,23.527313
BETA_SHORTTERM__10,-0.482968,2.093110
LAMBDA__1,3.760746,2.921830


## Building an estimated model with the `%%Makemymodel` magic

In a notebook the same workflow is more convenient with the `%%Makemymodel` magic, splitting the model over several cells with `segment=`. A useful naming convention: call the `Makemodel` instance `M<name>` — e.g. `Mdemo` — and extract the solvable `modelclass.model` instance as `<name> = M<name>.mmodel`.

In [13]:
%%Makemymodel Mdemo segment=consumption
## Private consumption
><stoc,estimator=ls,caption='Private consumption'> dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

## Private consumption
```text
><stoc,estimator=ls,caption='Private consumption'> dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)
```

**Estimation output**

| Item | Value |
|:--|:--|
| Estimator | ls |
| Effective sample | (np.int64(2012), np.int64(2019)) |
| Original equation | DLOG(CON) = -LAMBDA*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS(BETA_LONGTERM(1)))) + BETA_SHORTTERM(10)*DLOG(Y) |
| Estimated equation | DLOG(CON) = -(0.4416699417)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.8842598154)))) + (-0.0971117751)*DLOG(Y) |
| success | True |
| message | `gtol` termination condition is satisfied. |
| chisqr | 0.0008961636845 |
| redchi | 0.0001792327369 |
| aic | -66.77463216 |
| bic | -66.53630753 |

| Parameter | Estimate | t-stat |
|:--|--:|--:|
| BETA_LONGTERM__1 | 0.8842598154 | 30.389 |
| BETA_SHORTTERM__10 | -0.09711177512 | -0.483 |
| LAMBDA__1 | 0.4416699417 | 3.761 |

In [14]:
%%Makemymodel Mdemo segment=government
## Government expenditure
><stoc,estimator=ls> dlog(gov) = -lambda*(log(gov(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

## Government expenditure
```text
><stoc,estimator=ls> dlog(gov) = -lambda*(log(gov(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)
```

**Estimation output**

| Item | Value |
|:--|:--|
| Estimator | ls |
| Effective sample | (np.int64(2012), np.int64(2019)) |
| Original equation | DLOG(GOV) = -LAMBDA*(LOG(GOV(-1))-LOG(Y(-1))-LOG(ABS(BETA_LONGTERM(1)))) + BETA_SHORTTERM(10)*DLOG(Y) |
| Estimated equation | DLOG(GOV) = -(1.2457148604)*(LOG(GOV(-1))-LOG(Y(-1))-LOG(ABS((0.0756258858)))) + (1.6947092960)*DLOG(Y) |
| success | True |
| message | `gtol` termination condition is satisfied. |
| chisqr | 0.009063251913 |
| redchi | 0.001812650383 |
| aic | -48.26375067 |
| bic | -48.02542605 |

| Parameter | Estimate | t-stat |
|:--|--:|--:|
| BETA_LONGTERM__1 | 0.07562588582 | 23.527 |
| BETA_SHORTTERM__10 | 1.694709296 | 2.093 |
| LAMBDA__1 | 1.24571486 | 2.922 |

In [15]:
%%Makemymodel Mdemo segment=identities
## GDP identity
> <ident> y = con+inv+gov+exp-imp+ydisc

## GDP identity
```text
> <ident> y = con+inv+gov+exp-imp+ydisc
```

A final call *without* `segment=` combines all stored segments into the complete `Makemodel` instance. `render=False render_est=False` suppresses re-rendering of text and estimation output that has already been shown above.

In [16]:
%Makemymodel Mdemo render=False render_est=False
demo = Mdemo.mmodel

In minimum feedback order `k` must be an integer satisfying `0 < k < min(A.shape)`. 


## Add factors

Estimated equations only reproduce history when their error term — the **add factor** — is included. `.init_addfactors()` computes, for every behavioural equation, the add factors that make the model reproduce a given `DataFrame`, and returns a copy of the frame with the `_A` variables filled in. With `check=True` the resulting model solution is compared with the input data.

Add factors are covered in depth in the *Add factors* chapter.

In [17]:
baseline = Mdemo.init_addfactors(npl, 2016, 2024, check=True)



Difference between historic values and model results


,CON,GOV,Y
index,,,
2016,-4.656613e-10,0.000000e+00,-4.656613e-10
2017,-4.656613e-10,-4.074536e-10,-9.313226e-10
2018,-2.328306e-09,-3.201421e-10,-2.793968e-09
2019,-3.958121e-09,-9.313226e-10,-4.656613e-09
2020,-6.286427e-09,-1.164153e-10,-6.519258e-09
2021,-4.423782e-09,-1.746230e-10,-4.656613e-09
2022,-6.053597e-09,-5.820766e-10,-6.519258e-09
2023,-5.122274e-09,-1.164153e-10,-5.122274e-09
2024,-4.190952e-09,-2.910383e-10,-4.656613e-09


## Quick reference

```text
ls = Estimate_nls.with_defaults(smpl=(start, end), input_df=df,
                                param_names=[...], var_description={...})

est = ls("dlog(x) = -lambda*(log(x(-1))-log(y(-1))-log(abs(beta_longterm(1))))"
         " + beta_shortterm(10)*dlog(y) st. lambda=[0.1, 0.5]")
est.coef_ser; est.tvalues; est.org_eq_baked

mm = Makemodel("><stoc,estimator=ls,caption='...',smpl=2012 2019> dlog(x) = ...")
mm.render_est; mm.tvalues; mm.estimation_report(open_file=True); mm.report().show()
solvable = mm.mmodel
baseline = mm.init_addfactors(df, start, end, check=True)
```